In [1]:
import pandas as pd
import numpy as np

In [2]:
import os
os.chdir("D:\\Taiwei\\RestaurantBookings")
cwd = os.getcwd()

# order

In [3]:
order = pd.read_csv(
    './data/raw/train.csv', 
    #header = 1, 
    #names = ORDER_COLUMNS, 
    sep='\t',
    encoding='utf-16',
    on_bad_lines = 'warn'
    )

In [4]:
order.sample(n=10)

,booking_id,member_id,cdate,restaurant_id,datetime,people,purpose,gender,status,is_required_prepay_satisfied,return90
40395,57704,5817891a6f7ccb17533b53bafbb6605f31459e59,"12/8/2012,,10:55:00,PM",d584ebfc3e12c2f28bccc14b98a881432b7ce7dd,"12/13/2012,,5:30:00,PM",4,家人用餐,M,ok,1,0
100117,142824,da695f30de7f1fb195eee06f6fcad5a26f304d48,"6/10/2013,,1:22:00,PM",f88c23080c1959dd17d488ca75af168e10d0b31b,"6/10/2013,,7:00:00,PM",7,生日慶祝,M,new,1,0
40948,58473,593589644b972b1ad41099887c7f1c0e129dabc1,"4/10/2014,,9:29:00,PM",fe230aa38ebf31c978d0f58bd6fd5ab9e7ec1e66,"4/24/2014,,6:30:00,PM",2,朋友聚餐,M,ok,1,0
27604,39394,3c4594af3b037518836b3e1ccc0dd8830e4abe45,"8/6/2013,,7:24:00,PM",8a035d8ccfaae36efff1cdcc6f02f318f35abd64,"8/10/2013,,11:30:00,AM",4,家人用餐,F,new,1,0
11802,16904,19e1b5433b6accae4c24638427a554ef63998c33,"6/18/2012,,9:48:00,AM",bd76d1e31cb09d57afaa2c6393ec262085e1fade,"6/21/2012,,6:30:00,PM",4,生日慶祝,F,ok,1,0
13639,19516,1dfaaae9abb09ae1ed2b4d29b285e48a7c4d0ff0,"12/30/2013,,4:36:00,PM",1b95a47778f4856fc00c1ad2d05c1945e9c5aede,"1/6/2014,,6:30:00,PM",6,朋友聚餐,M,ok,1,0
109623,156500,ef654050114f3fe45785e1df9956596b7b1c9689,"1/11/2013,,5:59:00,PM",a4fd797fbb6edc57e212e3e4a8264571d6719429,"1/12/2013,,11:00:00,AM",5,朋友聚餐,F,canceled,1,0
74171,105845,a2136eb635dda734ba80d4a041009e53bd604914,"4/21/2014,,11:40:00,AM",e0ba53a1bcd2236253da3d4766b91c7b3f5e6f83,"5/2/2014,,7:30:00,PM",2,生日慶祝,M,canceled,1,1
26903,38373,3aba249cc046d3229a2ec5b5573501e05212cad3,"11/13/2013,,9:25:00,PM",8d6e0bcd450f5766b932bf7222f3427a293068e5,"11/15/2013,,7:00:00,PM",6,朋友聚餐,M,ok,1,0
35199,50259,4cca2e45ea4aeb74e60a4e9ee2e4eb2da45f0d3c,"9/27/2012,,7:48:00,PM",0a0275c0439bcad6baf71fcdb6374db1b98f1936,"9/28/2012,,1:30:00,PM",8,朋友聚餐,F,ok,1,0


In [5]:
order.dtypes

booking_id                       int64
member_id                       object
cdate                           object
restaurant_id                   object
datetime                        object
people                           int64
purpose                         object
gender                          object
status                          object
is_required_prepay_satisfied     int64
return90                         int64
dtype: object

In [6]:
conti_cols = ['people']
datetime_cols = ['cdate','datetime']
category_cols = ['purpose','gender','status']
binary_cols = ['is_required_prepay_satisfied']
ordinal_cols = []
y_cols = ['return90']

 ### 缺失值處理

In [7]:
from src.etl.utils import show_missing_values
show_missing_values(order)

,Amount of Missing Values,Percentage (%)
purpose,8235,7.040026


In [8]:
from src.etl.utils import fill_missing_values

In [9]:
col_name = 'purpose'
order1 = fill_missing_values(order, col_name = col_name, fill_value='Unknown')

### 重複值處理

In [10]:
from src.etl.utils import get_duplicates

get_duplicates(order1, subset = ['booking_id'])

,booking_id,member_id,cdate,restaurant_id,datetime,people,purpose,gender,status,is_required_prepay_satisfied,return90


### 時間格式轉換

In [11]:
from src.etl.utils import parse_datetime_column

In [12]:
# 分別處理 cdate (建立日期) 與 datetime (預約日期時間)
order2 = parse_datetime_column(order1, 'cdate')
order3 = parse_datetime_column(order2, 'datetime')

D:\Taiwei\RestaurantBookings\src\etl\utils.py:57: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean[col_name] = pd.to_datetime(df_clean[col_name], errors='coerce')
D:\Taiwei\RestaurantBookings\src\etl\utils.py:57: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean[col_name] = pd.to_datetime(df_clean[col_name], errors='coerce')


In [13]:
order3.head()

,booking_id,member_id,cdate,restaurant_id,datetime,people,purpose,gender,status,is_required_prepay_satisfied,return90
0,1,000036888f8cf9f0b9f931dee1536c6997bbbd27,2013-07-24 23:30:00,f3837c4489cc6c4587dfb8a92a060585c182ccc1,2013-08-08 19:00:00,5,家人用餐,F,ok,1,0
1,7,000462cdd95e750a50cd664021e037a5b471f5e2,2013-06-08 02:03:00,7c0079d1f506b211d53c00694577415061d78435,2013-06-09 18:00:00,9,朋友聚餐,M,ok,1,0
2,8,00049d1d216aa9674616749029571fecd453a42d,2012-10-08 11:44:00,7d818d2829ca905756e1d6b92efc07f904ef58a7,2012-10-19 18:30:00,2,甜蜜紀念日,M,ok,1,0
3,12,0005612efb88671ddcb49ae213e7fd5e03a48989,2014-01-22 19:59:00,e4b42c8e937bf0cdb01e4b90366358b70d5938ce,2014-01-24 18:30:00,2,生日慶祝,F,ok,1,1
4,13,0006018a813099f8e12c6e6c369c4efe90528f5c,2014-03-29 14:56:00,c5e3325051eab0d50ad2544e617d9a74ce433b2c,2014-03-29 17:30:00,4,生日慶祝,M,ok,1,1


### 類別變數處理

In [14]:
order3[category_cols[0]].value_counts()

purpose
朋友聚餐                                                41279
家人用餐                                                30656
生日慶祝                                                24611
甜蜜紀念日                                                8931
Unknown                                              8235
商務聚餐                                                 2534
重要約會                                                  445
請選擇                                                   142
紀念日慶祝                                                  82
其他                                                     28
Friends,Reunion                                        11
Please,Select                                           7
Family,Gathering                                        3
&#65533;&#40115;&#65533;&#65533;&#26813;&#65533;        2
Birthday                                                1
Friends                                                 1
家人用餐,盡量靠W                                               1
Busine

In [15]:
purpose_mapping = {
    '請選擇': 'Unknown',
    'Please,Select': 'Unknown',
    'Friends,Reunion': '朋友聚餐',
    'Family,Gathering': '家人用餐',       
    'Birthday': '生日慶祝',
    'Friends': '朋友聚餐',
    '家人用餐,盡量靠W': '朋友聚餐',
    'Business,meeting':'商務聚餐',
    '家人&#65533;':'家人用餐',
    '家人用餐-生日':'生日慶祝',
    '慶生':'生日慶祝',
    'Birthday,Celebration':'生日慶祝',
    '甜蜜紀念日': '紀念日慶祝'
}

In [16]:
from src.etl.utils import trim_whitespace, map_categorical_values
order4 = trim_whitespace(order3, "purpose")
order5 = map_categorical_values(order4, "purpose", mapping=purpose_mapping)
order6 = order5[order5['purpose'] != '&#65533;&#40115;&#65533;&#65533;&#26813;&#65533;']


In [17]:
order6['purpose'].value_counts()

purpose
朋友聚餐       41292
家人用餐       30660
生日慶祝       24615
紀念日慶祝       9013
Unknown     8384
商務聚餐        2535
重要約會         445
其他            28
Name: count, dtype: int64

In [18]:
order6.sample(n=5)

,booking_id,member_id,cdate,restaurant_id,datetime,people,purpose,gender,status,is_required_prepay_satisfied,return90
33651,48030,496f6894857efa0fc6955fcee9c3d03304ea5e16,2013-01-09 23:09:00,14087eb2baf89b7671c3bbd14feda0a9a9276b11,2013-01-12 21:00:00,5,朋友聚餐,M,ok,1,0
75476,107657,a4e815946e076e4367a61d27c426fbebf4bfa4d7,2012-10-31 12:33:00,3f47d2e07cb819918f27eb7237903685d4210374,2012-11-16 18:30:00,2,朋友聚餐,F,ok,1,0
9609,13743,151e36009f24c2c285e407f954b519271661530a,2013-05-27 08:53:00,95e6878c4f89e23c9975e8519a15c4d477385d18,2013-05-27 19:30:00,4,朋友聚餐,M,ok,1,0
75577,107788,a516103dff944f9ce95cc1455cf8ad20596870ce,2014-09-12 00:30:00,bb974967d1f63edbd841a954413027e9d9bf4903,2014-09-14 14:30:00,5,生日慶祝,F,ok,1,0
66278,94583,90cb3e9e309656936198ee3696517b17ac517536,2012-08-26 14:45:00,e62ec915c7340821572fa0d23e6f36d81e76af0e,2012-09-09 12:15:00,9,家人用餐,F,canceled,1,0


In [19]:
order6[category_cols[1]].value_counts()

gender
F    65008
M    51964
Name: count, dtype: int64

In [20]:
order6[category_cols[2]].value_counts()

status
ok          84355
canceled    22936
new          6243
no-show      1733
changed      1705
Name: count, dtype: int64

In [21]:
from src.etl.utils import convert_to_category

order7 = (order6
    .pipe(convert_to_category, col_name = category_cols[0])
    .pipe(convert_to_category, col_name = category_cols[1])
    .pipe(convert_to_category, col_name = category_cols[2])
)


In [23]:
from src.etl.utils import get_invalid_date_records
get_invalid_date_records(order7, created_at = 'cdate', reservation_at = 'datetime')

,booking_id,member_id,cdate,restaurant_id,datetime,people,purpose,gender,status,is_required_prepay_satisfied,return90


In [ ]:
#order7.to_csv('./data/interim/order_silver.csv')

# restaurant.csv

In [25]:
restaurant = pd.read_csv(
    './data/raw/restaurant_revised.csv', 
    #header = 1, 
    #names = ORDER_COLUMNS, 
    sep='\t',
    encoding='utf-16',
    on_bad_lines = 'warn'
    )

In [28]:
display(restaurant.sample(n=5))

,id,is_hotel,country,currency,city,cityarea,name,abbr,tel,opening_hours,...,outdoor_seating,wifi,wheelchair_accessible,price1,price2,lat,lng,timezone,locale,cdate
560,ecdf7239480dfe9307ec8972a0eb69fc187ffbe5,0,tw,TWD,台北市,NaN,3a18db23d6930b8cf9672bb9b362d5deee80da8f,3a18db23d6930b8cf9672bb9b362d5deee80da8f,74b1b457e7dc4bbc6e8cab46ab15171326740f87,11:30am-2:00am\r\n\r\n※ 用餐時間為1.5小時,...,0,1,0,400,600,25.04,121.55,Asia/Taipei,zh_TW,8/8/2013 13:15
511,ff5dd9f3f46420d596f4fc9b43d9f56be634950e,1,tw,TWD,台北市,請選擇,69610150c065a5b9113557f0d021e2a8b0ed290c,69610150c065a5b9113557f0d021e2a8b0ed290c,496b84f719c6c88617489d580cfc29a5025150df,午餐11:30~14:00\r\n晚餐17:30~21:30,...,0,0,1,1168,1905,25.05,121.54,Asia/Taipei,zh_TW,8/23/2012 15:33
507,97a0748a73da8a5bd360b11e6e2cc5321a83b14e,0,tw,TWD,台北市,內湖區,56732e7ce23ca54e7e4690569528812df758461a,56732e7ce23ca54e7e4690569528812df758461a,9806bbbc2097a0ec579890ecefd7ab6d33a1f55c,週一～週六 \r\n午餐：11:30-14:30\r\n晚餐：17:30-22:30\r\n...,...,1,1,1,473,991,25.06,121.58,Asia/Taipei,zh_TW,12/28/2010 16:13
178,20385a2efaeba9aa2f8461ad94c2fd7b0a6791ca,0,tw,TWD,新北市,請選擇,3316df9ca6be4f88d35a45e6cb400a23b5ac1f2d,3c77dc7499b86fba1bc4d3613814cb4605191904,9dd15bac3a9b7b72778c2676488de744fb70e81b,午餐:AM:11:20~14:00;晚餐:PM:17:20~21:00,...,0,0,1,400,700,25.03,121.47,Asia/Taipei,zh_TW,8/10/2009 14:57
363,8a369a36dbf4c871806025cf0f9329090f9fd4ec,1,tw,TWD,台北市,NaN,1d7a7679bb22ddc64e3138cef4d599106053f996,ea84c74360d2f286e1855fae9a4466e36934fafe,b02a2f522e63822e1536b3545d0c2828b9d7acbb,午餐 11:30 AM – 14:30 PM\r\n晚餐 18:00 PM – 22:00 ...,...,0,1,1,1000,1500,25.04,121.57,Asia/Taipei,zh_TW,3/7/2014 18:32


In [29]:
restaurant.dtypes

id                        object
is_hotel                   int64
country                   object
currency                  object
city                      object
cityarea                  object
name                      object
abbr                      object
tel                       object
opening_hours             object
good_for_family            int64
accept_credit_card         int64
parking                    int64
outdoor_seating            int64
wifi                       int64
wheelchair_accessible      int64
price1                     int64
price2                     int64
lat                      float64
lng                      float64
timezone                  object
locale                    object
cdate                     object
dtype: object

In [30]:
conti_cols = ['price1','price2']
datetime_cols = ['cdate']
category_cols = ['country','currency','city','cityarea','timezone','locale']
binary_cols = [
    'good_for_family','accept_credit_card','parking','outdoor_seating','wifi',
    'wheelchair_accessible'
    ]
ordinal_cols = []
unstructured_cols = ['opening_hours']
y_cols = []

In [31]:
show_missing_values(restaurant)

,Amount of Missing Values,Percentage (%)
cityarea,362,50.0


In [32]:
restaurant['cityarea'].sample(n=10)

564    請選擇
658     東區
92     中山區
23     松山區
569    NaN
296    請選擇
354    NaN
272    NaN
655    NaN
481    NaN
Name: cityarea, dtype: object

In [ ]:
rest2 = fill_missing_values(restaurant, 'cityarea',)

In [35]:
get_duplicates(rest2, 'id')

,id,is_hotel,country,currency,city,cityarea,name,abbr,tel,opening_hours,...,outdoor_seating,wifi,wheelchair_accessible,price1,price2,lat,lng,timezone,locale,cdate


In [36]:
get_duplicates(rest2, 'name')

,id,is_hotel,country,currency,city,cityarea,name,abbr,tel,opening_hours,...,outdoor_seating,wifi,wheelchair_accessible,price1,price2,lat,lng,timezone,locale,cdate


In [37]:
rest2[datetime_cols].sample(n=10)

,cdate
197,12/12/2012 12:43
136,12/23/2013 17:54
637,9/24/2013 15:45
502,4/8/2013 20:11
496,12/31/2009 0:00
109,12/31/2009 0:00
405,11/6/2012 1:12
646,5/12/2014 13:19
368,6/28/2012 21:55
371,8/25/2014 16:17


In [38]:
rest2[datetime_cols].dtypes

cdate    object
dtype: object

In [39]:
rest3 = parse_datetime_column(rest2, datetime_cols[0])

In [41]:
rest3[datetime_cols].head()

,cdate
0,2013-02-27 18:25:00
1,2014-08-26 13:37:00
2,2012-02-25 20:47:00
3,2012-06-25 11:59:00
4,2013-08-15 16:56:00


In [42]:
category_cols

['country', 'currency', 'city', 'cityarea', 'timezone', 'locale']

In [44]:
rest3[category_cols[0]].value_counts()

country
tw    716
hk      7
th      1
Name: count, dtype: int64

In [45]:
rest3[category_cols[1]].value_counts()

currency
TWD    716
HKD      7
THB      1
Name: count, dtype: int64

In [46]:
rest3[category_cols[2]].value_counts()

city
台北市    284
台中市     92
高雄市     82
新北市     57
台南市     50
上海市     48
桃園縣     39
新竹市     27
新竹縣     11
香港       7
基隆市      5
嘉義市      4
南投縣      4
宜蘭縣      3
苗栗縣      3
屏東縣      3
%        2
花蓮縣      2
雲林縣      1
Name: count, dtype: int64

In [48]:
city_mapping = {
    '桃園縣':'桃園市',
    '%':'Unknown'
}

In [49]:
rest4 = map_categorical_values(rest3, category_cols[2], mapping = city_mapping)

In [51]:
rest4[category_cols[2]].sample(n=8)

260    台北市
678    台中市
697    上海市
602    台中市
664    高雄市
573    台北市
500    台南市
705    台北市
Name: city, dtype: object

In [52]:
rest4[category_cols[3]].value_counts()

cityarea
Unknown    362
請選擇        185
中山區         30
大安區         23
信義區         14
中正區         14
中壢市         12
東區          11
士林區         10
松山區         10
魚池鄉          4
桃園市          4
萬華區          4
板橋區          3
烏日區          3
大同區          3
三民區          3
大園鄉          3
中和區          2
宜蘭市          2
花蓮市          2
內湖區          2
烏來區          2
南港區          1
竹南鎮          1
三重區          1
龜山鄉          1
新莊區          1
新店區          1
蘆洲區          1
淡水區          1
三義鄉          1
鳳山區          1
竹北市          1
旗山區          1
屏東市          1
西區           1
斗六市          1
八德市          1
Name: count, dtype: int64

In [53]:
rest4[category_cols[3]].unique()

array(['Unknown', '請選擇', '烏日區', '東區', '南港區', '松山區', '中山區', '大安區', '信義區',
       '士林區', '宜蘭市', '新莊區', '新店區', '蘆洲區', '淡水區', '三重區', '中壢市', '桃園市',
       '龜山鄉', '魚池鄉', '大園鄉', '竹南鎮', '中正區', '花蓮市', '大同區', '中和區', '板橋區',
       '竹北市', '烏來區', '鳳山區', '三義鄉', '三民區', '旗山區', '萬華區', '屏東市', '內湖區',
       '西區', '斗六市', '八德市'], dtype=object)

In [56]:
cityarea_mapping = {
    '請選擇':'Unknown'
}

In [57]:
rest5 = map_categorical_values(rest4, category_cols[3], mapping = cityarea_mapping)

In [60]:
rest5[category_cols[4]].value_counts()

timezone
Asia/Taipei    724
Name: count, dtype: int64

In [61]:
rest5[category_cols[5]].value_counts()

locale
zh_TW    708
en_US     16
Name: count, dtype: int64

In [63]:
rest6 = (rest5
    .pipe(convert_to_category, col_name=category_cols[0])
    .pipe(convert_to_category, col_name=category_cols[1])
    .pipe(convert_to_category, col_name=category_cols[2])
    .pipe(convert_to_category, col_name=category_cols[3])
    .pipe(convert_to_category, col_name=category_cols[4])
    .pipe(convert_to_category, col_name=category_cols[5])
)

## 異常值

In [68]:
rest6['price1'].describe()

count     724.000000
mean      571.281768
std       338.641919
min         0.000000
25%       354.750000
50%       500.000000
75%       676.250000
max      3000.000000
Name: price1, dtype: float64

In [69]:
rest5['price2'].describe()

count     724.000000
mean     1002.386740
std       718.524593
min         0.000000
25%       563.500000
50%       750.000000
75%      1127.750000
max      7000.000000
Name: price2, dtype: float64

In [ ]:
#rest6.to_csv('./data/interim/restaurant_silver.csv')

# member.csv

In [66]:
member = pd.read_csv(
    './data/raw/member.csv', 
    #header = 1, 
    #names = ORDER_COLUMNS, 
    sep='\t',
    encoding='utf-16',
    on_bad_lines = 'warn'
    )

In [73]:
member.sample(n=10)

,id,is_vip,gender,birthdate,city,has_google_id,has_yahoo_id,has_weibo_id,cdate
68001,e07de0bb6ab9867ad69ea33633d84a471ac5d256,0,F,0000-00-00,高雄市,0,0,0,8/31/2012 0:23
20462,c81e8190cbf329d5af9902f1625737f74ab5d91f,0,F,0000-00-00,台北市,0,0,0,6/11/2012 11:48
408871,d75d5bdfc0b3f1caa5950a298a1b6b40effb7ab3,0,M,10/24/1999,基隆市,0,0,0,11/14/2013 7:04
73149,023b175181f8ae272cd58326a5dc1432f7d6cc6b,0,F,0000-00-00,台北市,0,0,0,8/16/2012 16:31
394709,b42e663ad3f5bd95ef9bbdd3ad304840698528c8,0,F,0000-00-00,0,0,0,0,10/27/2013 8:21
243494,5463f0fccffa4ef19e78cdee3f7061d851a6814f,0,M,0000-00-00,0,0,0,0,3/19/2013 21:50
161850,7547ea83af27e1e8aac9114497633578b05235ca,0,M,11/10/1982,0,0,0,0,11/16/2012 16:54
568907,45eb836f8f0b9dc5c71c422b8276ca64559ba6df,0,M,0000-00-00,0,0,0,0,5/19/2014 16:56
531310,b6b6db4401aac1977108e239ee32ce40eb8e967c,0,F,7/16/1984,新北市,0,0,0,4/7/2014 12:05
181980,afae5619e4df1b1f49bd758f30a331a3f7d2f591,0,F,0000-00-00,0,0,0,0,12/13/2012 16:12


In [70]:
member.dtypes

id               object
is_vip            int64
gender           object
birthdate        object
city             object
has_google_id     int64
has_yahoo_id      int64
has_weibo_id      int64
cdate            object
dtype: object

In [95]:
conti_cols = []
datetime_cols = ['birthdate','cdate']
category_cols = ['gender','city']
binary_cols = [
    'is_vip','has_google_id','has_yahoo_id','has_weibo_id'
    ]
ordinal_cols = []
unstructured_cols = []
y_cols = []

In [75]:
show_missing_values(member)

,Amount of Missing Values,Percentage (%)
city,478,0.068364
gender,42,0.006007


In [76]:
member2 = (member
    .pipe(fill_missing_values, col_name = 'city')
    .pipe(fill_missing_values, col_name = 'gender')
)

In [78]:
get_duplicates(member2, subset = ['id'])

,id,is_vip,gender,birthdate,city,has_google_id,has_yahoo_id,has_weibo_id,cdate


In [83]:
member3 = (member2
    .pipe(parse_datetime_column, col_name = datetime_cols[0])
    .pipe(parse_datetime_column, col_name = datetime_cols[1])
)

In [85]:
member3[datetime_cols].sample(n=10)

,birthdate,cdate
568828,NaT,2014-05-19 15:46:00
586487,NaT,2014-06-06 20:50:00
603100,NaT,2014-06-23 12:26:00
100527,NaT,2012-05-21 06:13:00
166687,NaT,2012-11-23 17:53:00
355979,NaT,2013-09-05 10:43:00
233842,NaT,2013-03-03 21:23:00
250327,1992-07-15,2013-04-01 21:50:00
268187,NaT,2013-04-29 20:41:00
439687,NaT,2013-12-17 23:30:00


In [84]:
member3[datetime_cols].describe()

,birthdate,cdate
count,187745,699200
mean,1982-02-20 20:38:53.590774720,2013-07-23 22:46:15.269536768
min,1900-01-01 00:00:00,2012-01-01 00:13:00
25%,1977-02-02 00:00:00,2012-12-04 23:31:45
50%,1983-04-29 00:00:00,2013-08-27 20:25:30
75%,1988-11-12 00:00:00,2014-03-29 10:21:00
max,2014-12-24 00:00:00,2014-09-23 17:02:00


In [86]:
category_cols

['gender', 'city', 'city', 'cityarea', 'timezone', 'locale']

In [87]:
member3[category_cols[0]].value_counts()


gender
F          377728
M          321430
Unknown        42
Name: count, dtype: int64

In [88]:
member3[category_cols[1]].value_counts()

city
0                                   500357
台北市                                  70389
高雄市                                  34270
新北市                                  20671
台中市                                  20110
基隆市                                  11203
桃園縣                                   9698
台南市                                   8778
其他                                    6850
新竹市                                   4949
新竹縣                                   2798
屏東縣                                   1512
彰化縣                                   1320
苗栗縣                                   1286
上海市                                    988
南投縣                                    855
雲林縣                                    653
嘉義市                                    581
花蓮縣                                    483
Unknown                                478
宜蘭縣                                    381
香港                                     247
嘉義縣                                    200
台東縣   

In [91]:
member_city_mapping = {
    '桃園縣':'桃園市',
    '0':'Unknown',
    0:'Unknown'
}

In [92]:
member4 = member3[member3['city'] != '??蝮?>??蝮?/option><option value']
member4 = map_categorical_values(member4, 'city', mapping = member_city_mapping)

In [93]:
member4['city'].value_counts()

city
Unknown    500835
台北市         70389
高雄市         34270
新北市         20671
台中市         20110
基隆市         11203
桃園市          9698
台南市          8778
其他           6850
新竹市          4949
新竹縣          2798
屏東縣          1512
彰化縣          1320
苗栗縣          1286
上海市           988
南投縣           855
雲林縣           653
嘉義市           581
花蓮縣           483
宜蘭縣           381
香港            247
嘉義縣           200
台東縣            87
澎湖縣            27
金門縣            23
連江縣             5
Name: count, dtype: int64

In [101]:
member5 = (member4
    .pipe(convert_to_category, col_name = category_cols[0])
    .pipe(convert_to_category, col_name = category_cols[1])
)

In [96]:
binary_cols

['is_vip', 'has_google_id', 'has_yahoo_id', 'has_weibo_id']

In [102]:
member5[binary_cols[0]].value_counts()

is_vip
0    699137
1        62
Name: count, dtype: int64

In [103]:
member5[binary_cols[1]].value_counts()

has_google_id
0    698735
1       464
Name: count, dtype: int64

In [104]:
member5[binary_cols[2]].value_counts()

has_yahoo_id
0    698667
1       532
Name: count, dtype: int64

In [105]:
member5[binary_cols[3]].value_counts()

has_weibo_id
0    699111
1        88
Name: count, dtype: int64

In [106]:
member5.to_csv('./data/interim/member_silver.csv')